
# Weather Oracle Research Lab

## Context Contract
- Crypto technical indicators, crypto execution logic, and momentum features are out of scope for this notebook.
- This notebook is the canonical research artifact for validating Weather Oracle alpha before any orchestrator integration.
- Authoritative references for this refactor:
  - `Weather/APIs/OpenMeteo_Ensemble_API.md`
  - `Weather/Settlement/NWS_Settlement_Rules.md`
  - `Weather/Features/weather_features_schema.md`
- Working-memory hygiene is governed by `.agent/rules/context_hygiene.md`.
- Architectural decisions made here must be offloaded to `.agent/buffer/session_logs.md`.



## Canonical Pipeline
1. Environment and city/station constants
2. Open-Meteo ensemble forecast ingestion
3. NWS ground-truth ingestion and settlement-day normalization
4. Feature engineering for daily high-temperature exceedance
5. Deterministic market simulation and scorecarding
6. Threshold analysis and research summary


In [ ]:

import math
from datetime import UTC, datetime, timedelta
from statistics import NormalDist
from typing import Any

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

plt.style.use('fivethirtyeight')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

EDGE_FLOOR = 0.08
DEFAULT_CITY = 'Chicago'
DEFAULT_START_DATE = '2026-03-20'
DEFAULT_END_DATE = '2026-04-08'
DEFAULT_ENSEMBLE_MODEL = 'gfs_seamless'

CITY_CONFIG = {
    'Chicago': {
        'latitude': 41.98,
        'longitude': -87.90,
        'station_id': 'KORD',
        'station_suffix': 'ORD',
        'timezone': 'America/Chicago',
        'wfo': 'lot',
        'climatology_avg_high_f': 58.0,
    },
    'New York City': {
        'latitude': 40.78,
        'longitude': -73.97,
        'station_id': 'KNYC',
        'station_suffix': 'NYC',
        'timezone': 'America/New_York',
        'wfo': 'okx',
        'climatology_avg_high_f': 60.0,
    },
    'Washington DC': {
        'latitude': 38.85,
        'longitude': -77.04,
        'station_id': 'KDCA',
        'station_suffix': 'DCA',
        'timezone': 'America/New_York',
        'wfo': 'lwx',
        'climatology_avg_high_f': 63.0,
    },
}

CLI_PUBLICATION_NOTE = (
    'CLI products are preliminary and often published overnight or in early-morning windows after the local observation day.'
)


In [ ]:

OPEN_METEO_ENSEMBLE_URL = 'https://ensemble-api.open-meteo.com/v1/ensemble'
NWS_PRODUCTS_API = 'https://api.weather.gov/products'
NWS_OBSERVATIONS_API = 'https://api.weather.gov/stations/{station_id}/observations'
REQUEST_HEADERS = {'User-Agent': 'WeatherOracleResearch/1.0 (sigey local research notebook)'}


def _to_fahrenheit(value_c: pd.Series | np.ndarray | float) -> Any:
    return (np.asarray(value_c, dtype=float) * 9.0 / 5.0) + 32.0


def _normal_cdf(x: float) -> float:
    return NormalDist().cdf(x)


def probability_of_exceedance(strike_f: float, mean_f: float, std_f: float, min_std: float = 0.5) -> float:
    sigma = max(float(std_f), min_std)
    z_value = (float(strike_f) - float(mean_f)) / sigma
    return 1.0 - _normal_cdf(z_value)


def _safe_skew(values: np.ndarray) -> float:
    arr = np.asarray(values, dtype=float)
    if arr.size < 3:
        return 0.0
    mean = arr.mean()
    std = arr.std(ddof=0)
    if std == 0:
        return 0.0
    return float(np.mean(((arr - mean) / std) ** 3))


def _member_columns(frame: pd.DataFrame, prefix: str) -> list[str]:
    candidates = [col for col in frame.columns if str(col).startswith(prefix)]
    return sorted(candidates)


In [ ]:

def fetch_openmeteo_ensemble(city: str = DEFAULT_CITY, start_date: str = DEFAULT_START_DATE, end_date: str = DEFAULT_END_DATE, model: str = DEFAULT_ENSEMBLE_MODEL) -> tuple[pd.DataFrame, dict[str, Any]]:
    city_meta = CITY_CONFIG[city]
    params = {
        'latitude': city_meta['latitude'],
        'longitude': city_meta['longitude'],
        'hourly': 'temperature_2m',
        'temperature_unit': 'fahrenheit',
        'timezone': city_meta['timezone'],
        'models': model,
    }
    if start_date and end_date:
        params['start_date'] = start_date
        params['end_date'] = end_date
    else:
        params['forecast_days'] = 16
        params['past_days'] = 16

    response = requests.get(OPEN_METEO_ENSEMBLE_URL, params=params, headers=REQUEST_HEADERS, timeout=60)
    try:
        response.raise_for_status()
    except requests.HTTPError as exc:
        raise RuntimeError(f"Open-Meteo ensemble request failed: {response.status_code} {response.text[:250]}") from exc
    payload = response.json()
    hourly = payload.get('hourly') or {}
    if not hourly:
        raise ValueError('Open-Meteo ensemble payload did not include hourly data.')

    frame = pd.DataFrame(hourly)
    if 'time' not in frame.columns:
        raise ValueError('Open-Meteo ensemble payload is missing hourly.time.')

    frame['forecast_time_local'] = pd.to_datetime(frame['time'])
    frame = frame.drop(columns=['time'])

    temp_member_cols = [col for col in frame.columns if str(col).startswith('temperature_2m')]
    if not temp_member_cols:
        raise ValueError('Open-Meteo ensemble payload did not expose any temperature_2m member fields.')

    renamed = {}
    member_index = 0
    for col in temp_member_cols:
        if str(col) == 'temperature_2m':
            renamed[col] = 'member_00'
            continue
        renamed[col] = f'member_{member_index:02d}'
        member_index += 1
    frame = frame.rename(columns=renamed)
    member_cols = _member_columns(frame, 'member_')

    metadata = {
        'city': city,
        'station_id': city_meta['station_id'],
        'timezone': payload.get('timezone') or city_meta['timezone'],
        'ensemble_model': model,
        'member_count': len(member_cols),
        'generationtime_ms': payload.get('generationtime_ms'),
        'source': 'open-meteo-ensemble',
        'dispersion_mode': 'member-level' if len(member_cols) > 1 else 'single-series-fallback',
        'research_limitations': 'If the API exposes only a single temperature trace, ensemble dispersion is approximated downstream via the minimum sigma guard.',
        'fetch_timestamp_utc': datetime.now(UTC).isoformat(),
    }
    return frame[['forecast_time_local', *member_cols]], metadata


def aggregate_daily_high_ensemble(hourly_frame: pd.DataFrame, metadata: dict[str, Any]) -> pd.DataFrame:
    member_cols = _member_columns(hourly_frame, 'member_')
    if not member_cols:
        raise ValueError('No ensemble member columns available for daily aggregation.')

    frame = hourly_frame.copy()
    frame['settlement_day_local'] = frame['forecast_time_local'].dt.tz_localize(None).dt.floor('D')
    daily_highs = frame.groupby('settlement_day_local')[member_cols].max().sort_index()
    daily_highs.index.name = 'settlement_day_local'

    records = []
    fetch_hour_local = pd.Timestamp(metadata['fetch_timestamp_utc']).tz_convert(metadata['timezone']).hour
    rolling_mean = daily_highs.mean(axis=1).rolling(window=14, min_periods=3).mean()

    for day, row in daily_highs.iterrows():
        values = row.to_numpy(dtype=float)
        ensemble_mean = float(np.mean(values))
        ensemble_std = float(np.std(values, ddof=0)) if len(values) > 1 else 0.0
        records.append(
            {
                'city': metadata['city'],
                'station_id': metadata['station_id'],
                'forecast_issue_time_utc': metadata['fetch_timestamp_utc'],
                'settlement_day_local': pd.Timestamp(day).tz_localize(None),
                'ensemble_mean': ensemble_mean,
                'ensemble_std': ensemble_std,
                'ensemble_skew': _safe_skew(values),
                'temp_drift_from_avg': ensemble_mean - float(rolling_mean.loc[day]) if not pd.isna(rolling_mean.loc[day]) else 0.0,
                'hour_of_forecast': fetch_hour_local,
                'member_count': metadata['member_count'],
                'dispersion_mode': metadata['dispersion_mode'],
            }
        )
    return pd.DataFrame(records)


In [ ]:


def fetch_nws_station_observations(city: str = DEFAULT_CITY, start_date: str = DEFAULT_START_DATE, end_date: str = DEFAULT_END_DATE) -> pd.DataFrame:
    city_meta = CITY_CONFIG[city]
    station_id = city_meta['station_id']
    start = pd.Timestamp(start_date, tz=city_meta['timezone']).tz_convert('UTC')
    end = (pd.Timestamp(end_date, tz=city_meta['timezone']) + pd.Timedelta(days=1)).tz_convert('UTC')

    observations: list[dict[str, Any]] = []
    next_url: str | None = None
    while True:
        if next_url:
            response = requests.get(next_url, headers=REQUEST_HEADERS, timeout=30)
        else:
            params = {
                'start': start.isoformat().replace('+00:00', 'Z'),
                'end': end.isoformat().replace('+00:00', 'Z'),
                'limit': 500,
            }
            response = requests.get(
                NWS_OBSERVATIONS_API.format(station_id=station_id),
                params=params,
                headers=REQUEST_HEADERS,
                timeout=30,
            )
        response.raise_for_status()
        payload = response.json()
        features = payload.get('features') or []
        for feature in features:
            props = feature.get('properties') or {}
            observations.append(props)
        next_url = payload.get('pagination', {}).get('next')
        if not next_url:
            break

    if not observations:
        raise ValueError(f'No NWS observations returned for station {station_id}.')

    obs = pd.DataFrame(observations)
    obs['timestamp_utc'] = pd.to_datetime(obs['timestamp'], utc=True)
    obs['timestamp_local'] = obs['timestamp_utc'].dt.tz_convert(city_meta['timezone'])
    obs['temperature_c'] = obs['temperature'].apply(lambda item: item.get('value') if isinstance(item, dict) else None)
    obs = obs.dropna(subset=['temperature_c']).copy()
    obs['temperature_f'] = _to_fahrenheit(obs['temperature_c'])
    obs['settlement_day_local'] = obs['timestamp_local'].dt.tz_localize(None).dt.floor('D')
    return obs[['timestamp_utc', 'timestamp_local', 'settlement_day_local', 'temperature_f']].sort_values('timestamp_utc')


def build_ground_truth_from_nws(city: str = DEFAULT_CITY, start_date: str = DEFAULT_START_DATE, end_date: str = DEFAULT_END_DATE) -> pd.DataFrame:
    city_meta = CITY_CONFIG[city]
    observations = fetch_nws_station_observations(city=city, start_date=start_date, end_date=end_date)
    truth = (
        observations.groupby('settlement_day_local', as_index=False)
        .agg(observed_high_f=('temperature_f', 'max'))
        .sort_values('settlement_day_local')
    )
    truth['station_id'] = city_meta['station_id']
    truth['settlement_source'] = 'NWS_OBSERVATIONS_PROXY_FOR_CLI'
    truth['report_publication_note'] = CLI_PUBLICATION_NOTE
    return truth



> **Ground-truth note**
>
> This notebook uses official NWS station observations grouped to the local settlement day as the executable label path in research mode. CLI/CF6 remain the settlement authority, and the report publication lag is modeled explicitly in the metadata and summary. If you later wire true CLI/CF6 parsers, they should replace the observation-proxy label source without changing the downstream feature or scorecard contract.


In [ ]:


def build_strike_ladder(features: pd.DataFrame, increment_f: int = 2) -> pd.DataFrame:
    rows = []
    for record in features.to_dict(orient='records'):
        mean_f = float(record['ensemble_mean'])
        std_f = float(record['ensemble_std'])
        if not np.isfinite(mean_f):
            continue
        if not np.isfinite(std_f):
            std_f = 0.0
        base_strike = increment_f * round(mean_f / increment_f)
        strikes = [base_strike - 4, base_strike - 2, base_strike, base_strike + 2, base_strike + 4]
        for strike in strikes:
            strike = float(strike)
            model_poe = probability_of_exceedance(strike, mean_f, std_f)
            rows.append({**record, 'ensemble_std': std_f, 'strike_f': strike, 'model_poe': model_poe})
    return pd.DataFrame(rows)


def simulate_implied_probability(row: pd.Series) -> float:
    scaled_distance = (float(row['strike_f']) - float(row['ensemble_mean'])) / max(float(row['ensemble_std']), 1.5)
    baseline = 1.0 / (1.0 + math.exp(scaled_distance))
    structural_bias = 0.02 * np.tanh(float(row['temp_drift_from_avg']) / 8.0)
    forecast_hour_bias = 0.01 * np.cos(float(row['hour_of_forecast']) / 24.0 * 2 * math.pi)
    implied = np.clip(baseline + structural_bias + forecast_hour_bias, 0.02, 0.98)
    return float(implied)


def build_backtest_dataset(city: str = DEFAULT_CITY, start_date: str = DEFAULT_START_DATE, end_date: str = DEFAULT_END_DATE) -> tuple[pd.DataFrame, dict[str, Any]]:
    hourly_ensemble, ensemble_meta = fetch_openmeteo_ensemble(city=city, start_date=start_date, end_date=end_date)
    features = aggregate_daily_high_ensemble(hourly_ensemble, ensemble_meta)
    features = features.dropna(subset=['ensemble_mean']).copy()
    strikes = build_strike_ladder(features)
    truth = build_ground_truth_from_nws(city=city, start_date=start_date, end_date=end_date)

    dataset = strikes.merge(truth, on=['settlement_day_local', 'station_id'], how='inner')
    if dataset.empty:
        metadata = {
            **ensemble_meta,
            'edge_floor': EDGE_FLOOR,
            'label_source': 'unavailable',
            'dispersion_mode': features['dispersion_mode'].iloc[0] if not features.empty else ensemble_meta['dispersion_mode'],
        }
        return dataset, metadata
    dataset['actual_yes'] = (dataset['observed_high_f'] > dataset['strike_f']).astype(int)
    dataset['kalshi_implied_prob'] = dataset.apply(simulate_implied_probability, axis=1)
    dataset['edge'] = dataset['model_poe'] - dataset['kalshi_implied_prob']
    dataset['abs_edge'] = dataset['edge'].abs()
    dataset['qualifies_trade'] = dataset['abs_edge'] > EDGE_FLOOR
    dataset['side'] = np.where(dataset['edge'] >= 0, 'YES', 'NO')

    yes_return = dataset['actual_yes'] - dataset['kalshi_implied_prob']
    no_return = (1 - dataset['actual_yes']) - (1 - dataset['kalshi_implied_prob'])
    dataset['virtual_return'] = np.where(dataset['side'] == 'YES', yes_return, no_return)
    dataset['virtual_return'] = np.where(dataset['qualifies_trade'], dataset['virtual_return'], 0.0)
    dataset['brier_component'] = (dataset['model_poe'] - dataset['actual_yes']) ** 2

    metadata = {
        **ensemble_meta,
        'edge_floor': EDGE_FLOOR,
        'label_source': dataset['settlement_source'].iloc[0] if not dataset.empty else 'unavailable',
        'dispersion_mode': features['dispersion_mode'].iloc[0] if not features.empty else ensemble_meta['dispersion_mode'],
    }
    return dataset, metadata


In [ ]:


def score_shadow_backtest(dataset: pd.DataFrame) -> dict[str, Any]:
    qualified = dataset.loc[dataset['qualifies_trade']].copy()
    overall_brier = float(dataset['brier_component'].mean()) if not dataset.empty else float('nan')
    qualified_brier = float(qualified['brier_component'].mean()) if not qualified.empty else float('nan')
    hit_rate = float((qualified['virtual_return'] > 0).mean()) if not qualified.empty else float('nan')

    threshold_rows = []
    for threshold in np.arange(0.50, 0.91, 0.05):
        bucket = dataset.loc[(dataset['model_poe'] >= threshold) & dataset['qualifies_trade']].copy()
        if bucket.empty:
            continue
        threshold_rows.append(
            {
                'poe_threshold': round(float(threshold), 2),
                'trade_count': int(len(bucket)),
                'virtual_pnl': float(bucket['virtual_return'].sum()),
                'avg_return': float(bucket['virtual_return'].mean()),
                'brier_score': float(bucket['brier_component'].mean()),
            }
        )
    threshold_table = pd.DataFrame(threshold_rows)

    optimal_threshold = None
    if not threshold_table.empty:
        profitable = threshold_table.loc[threshold_table['virtual_pnl'] > 0].sort_values(['virtual_pnl', 'avg_return'], ascending=False)
        if not profitable.empty:
            optimal_threshold = profitable.iloc[0].to_dict()

    return {
        'total_opportunities': int(len(dataset)),
        'qualified_trades': int(dataset['qualifies_trade'].sum()),
        'hit_rate': hit_rate,
        'overall_brier': overall_brier,
        'qualified_brier': qualified_brier,
        'virtual_pnl': float(dataset['virtual_return'].sum()),
        'threshold_table': threshold_table,
        'optimal_threshold': optimal_threshold,
    }


In [1]:

CITY = DEFAULT_CITY
START_DATE = DEFAULT_START_DATE
END_DATE = DEFAULT_END_DATE

weather_backtest_df, weather_meta = build_backtest_dataset(city=CITY, start_date=START_DATE, end_date=END_DATE)
scorecard = score_shadow_backtest(weather_backtest_df)

print('Data source:', weather_meta['source'])
print('Dispersion mode:', weather_meta['dispersion_mode'])
print('Label source:', weather_meta['label_source'])
print('Total opportunities:', scorecard['total_opportunities'])
print('Qualified trades:', scorecard['qualified_trades'])
print('Overall Brier Score:', round(scorecard['overall_brier'], 4))
print('Qualified Brier Score:', round(scorecard['qualified_brier'], 4) if not np.isnan(scorecard['qualified_brier']) else 'n/a')
print('Virtual PnL:', round(scorecard['virtual_pnl'], 4))

weather_backtest_df.head()


Data source: open-meteo-ensemble
Dispersion mode: member-level
Label source: NWS_OBSERVATIONS_PROXY_FOR_CLI
Total opportunities: 20
Qualified trades: 13
Overall Brier Score: 0.2377
Qualified Brier Score: 0.3123
Virtual PnL: -2.0961


In [ ]:

feature_builder_columns = [
    'settlement_day_local',
    'ensemble_mean',
    'ensemble_std',
    'ensemble_skew',
    'temp_drift_from_avg',
    'hour_of_forecast',
    'strike_f',
    'model_poe',
    'actual_yes',
    'kalshi_implied_prob',
    'edge',
]

weather_backtest_df[feature_builder_columns].head(10)


In [2]:

threshold_table = scorecard['threshold_table'].copy()
threshold_table


,poe_threshold,trade_count,virtual_pnl,avg_return,brier_score
0,0.50,6,1.361037,0.226840,0.021682
1,0.55,6,1.361037,0.226840,0.021682
2,0.60,6,1.361037,0.226840,0.021682
3,0.65,6,1.361037,0.226840,0.021682
4,0.70,6,1.361037,0.226840,0.021682
5,0.75,5,1.015504,0.203101,0.012342
6,0.80,5,1.015504,0.203101,0.012342
7,0.85,3,0.427873,0.142624,0.001411
8,0.90,3,0.427873,0.142624,0.001411


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
weather_backtest_df['brier_component'].plot(kind='hist', bins=20, ax=axes[0], title='Brier Component Distribution')
axes[0].set_xlabel('Brier component')

weather_backtest_df.loc[weather_backtest_df['qualifies_trade'], 'virtual_return'].cumsum().plot(ax=axes[1], title='Qualified Virtual PnL')
axes[1].set_ylabel('Cumulative virtual return')
plt.tight_layout()
plt.show()


In [3]:

optimal = scorecard['optimal_threshold']
if optimal is None:
    print('No profitable PoE threshold found under the current simulation assumptions.')
else:
    print('Optimal threshold summary')
    print(pd.Series(optimal))


Optimal threshold summary
poe_threshold    0.500000
trade_count      6.000000
virtual_pnl      1.361037
avg_return       0.226840
brier_score      0.021682
dtype: float64



## Research Summary Template
- **Best threshold:** determined from the threshold analysis cell above
- **Brier Score:** use `overall_brier` as the primary calibration metric
- **Trade count:** use `qualified_trades`
- **Virtual PnL:** use `virtual_pnl`
- **Known caveats:**
  - current label path uses official NWS station observations as an executable proxy until direct CLI/CF6 parsing is added
  - Open-Meteo member-field availability may require a dispersion fallback depending on endpoint behavior
  - simulated implied probabilities are a research proxy, not historical Kalshi order book data
